In [55]:
--Adolfo David Romero
--991555778
--Assignment 3B (bonus)
USE section19
GO

Commands completed successfully.

Total execution time: 00:00:00.003

In [59]:
--Verify and test tables for testing (copy-pasted from previous assingnment)
--Drop tables to start fresh and create tables
DROP TABLE IF EXISTS OrderProd;
DROP TABLE IF EXISTS SalesOrder;
DROP TABLE IF EXISTS Customer;

--CREATE tables with rows mentioned in assignment
CREATE TABLE Customer(
    custno INT PRIMARY KEY, 
    custname VARCHAR(50),
    balance DECIMAL(10,2)
);
CREATE TABLE SalesOrder(
    orderno INT PRIMARY KEY, 
    custno INT,
    orderdate DATE, 
    FOREIGN KEY (custno) REFERENCES CUSTOMER(custno)
);
CREATE TABLE OrderProd(
    orderno INT, 
    partno INT,
    orderqty INT, 
    orderprice DECIMAL(10,2),
    PRIMARY KEY (orderno, partno),
    FOREIGN KEY (orderno) REFERENCES SalesOrder(orderno)
);

--LOAD tables with fake data
INSERT INTO Customer (custno, custname, balance) VALUES 
(1, 'David Romero', 20.00), 
(2, 'Robert Marley', 400.00), 
(3, 'Brent Perteron', 00.00), 
(4, 'Magdin Stoica', 8045.57), 
(5, 'Bob Bobbert', 94.84),
(6, 'Testy McGee', 0.00); --Test case

INSERT INTO SalesOrder (orderno, custno, orderdate) VALUES 
(1,1,'2025-01-01'),
(2,2,'2025-01-01'),
(3,2,'2025-03-11'),
(4,3,'1999-02-19'),
(5,5,'2025-01-03')

INSERT INTO OrderProd (orderno, partno, orderqty, orderprice) VALUES
(1, 1, 2, 10.00),  
(2, 2, 1, 200.00), 
(3, 3, 5, 30.00),  
(4, 4, 1, 10.00),  
(5, 5, 3, 15.00);  

--Confirm tables
SELECT * FROM Customer
SELECT * FROM SalesOrder
SELECT * FROM OrderProd

(6 rows affected)

(5 rows affected)

(5 rows affected)

(6 rows affected)

(5 rows affected)

(5 rows affected)

Total execution time: 00:00:00.058

custno,custname,balance
1,David Romero,20.00
2,Robert Marley,400.00
3,Brent Perteron,0.00
4,Magdin Stoica,8045.57
5,Bob Bobbert,94.84
6,Testy McGee,0.00


orderno,custno,orderdate
1,1,2025-01-01
2,2,2025-01-01
3,2,2025-03-11
4,3,1999-02-19
5,5,2025-01-03


orderno,partno,orderqty,orderprice
1,1,2,10.00
2,2,1,200.00
3,3,5,30.00
4,4,1,10.00
5,5,3,15.00


# **PART A - CURSORS**

In [60]:
-- 1
DROP PROCEDURE AdjustAccountBalances
GO

CREATE PROCEDURE AdjustAccountBalances
AS 
BEGIN
    DECLARE @custno INT, @totalAmount DECIMAL(10,2); -- Store customer number and total transaction amount 
    DECLARE customer_cursor CURSOR FOR SELECT custno FROM Customer --customer cursor 

    OPEN customer_cursor; --executes above 'SELECT custno FROM Customer'
    FETCH NEXT FROM customer_cursor INTO @custno; --fetch the first customer

    WHILE @@FETCH_STATUS = 0 --loop through all customers (Iterate through each customer's sales transactions.)
    BEGIN

        --calculates total sales for the customer, cost per product
        SELECT @totalAmount = ISNULL(SUM(op.orderqty * op.orderprice), 0) -- ISNULL is used in case cust has NO orders (potential edge case). Resurns 0 if null
        FROM SalesOrder so --use alliases to acccess
        JOIN ORDERPROD op ON so.orderno = op.orderno
        WHERE so.custno = @custno

        PRINT CONCAT('UPDATED TOTAL AMOUNT: ',@totalAmount);

        --update customer balance col
        UPDATE CUSTOMER
        SET balance = balance - @totalAmount --subtract balance using var
        WHERE custno = @custno 

        PRINT CONCAT('UPDATED CUSTOMER: ',@custno);

        FETCH NEXT FROM customer_cursor INTO @custno; -- Move to next customer
    END;

    --clean up crew
    CLOSE customer_cursor;
    DEALLOCATE customer_cursor;

END;

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.021

In [61]:
--Test Cursor

--Customer 6 has an initial balance of 0 for a test case
SELECT * FROM Customer --before
SELECT * FROM SalesOrder
SELECT * FROM OrderProd

EXEC AdjustAccountBalances;

SELECT * FROM Customer --after


(6 rows affected)

(5 rows affected)

(5 rows affected)

UPDATED TOTAL AMOUNT: 20.00

(1 row affected)

...

Total execution time: 00:00:00.041

custno,custname,balance
1,David Romero,20.00
2,Robert Marley,400.00
3,Brent Perteron,0.00
4,Magdin Stoica,8045.57
5,Bob Bobbert,94.84
6,Testy McGee,0.00


orderno,custno,orderdate
1,1,2025-01-01
2,2,2025-01-01
3,2,2025-03-11
4,3,1999-02-19
5,5,2025-01-03


orderno,partno,orderqty,orderprice
1,1,2,10.00
2,2,1,200.00
3,3,5,30.00
4,4,1,10.00
5,5,3,15.00


custno,custname,balance
1,David Romero,0.00
2,Robert Marley,50.00
3,Brent Perteron,-10.00
4,Magdin Stoica,8045.57
5,Bob Bobbert,49.84
6,Testy McGee,0.00


# **PART B - DYNAMIC SQL**